<a href="https://colab.research.google.com/github/FaarisIq/Persuasion-Analysis-Engine/blob/main/persuasion_analysis_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Persuasion Analysis Engine - Faaris Iqbal

In [1]:
!python -m spacy download en_core_web_sm
!pip install praw pandas spacy vaderSentiment

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 10.2 MB/s eta 0:00:00


In [9]:
import pandas as pd
import re
import spacy
import json
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import numpy as np
from datetime import datetime


"""
Persuasion Analysis Engine - Faaris Iqbal

Analyzes persuasiveness in r/ChangeMyView by scoring posts based on the
quality of their comment threads, where actual persuasion (deltas) happens.
"""


nlp = spacy.load("en_core_web_sm")
analyzer = SentimentIntensityAnalyzer()

SCORING_WEIGHTS = {
    'top_comment_quality': 0.30,
    'discussion_depth': 0.20,
    'evidence_quality': 0.15,
    'engagement_quality': 0.15,
    'sophistication': 0.10,
    'emotion_balance': 0.10
}


def safe_get(data, key, default=0.0):
    if hasattr(data, 'get'):
        value = data.get(key, default)
    else:
        value = default
    if pd.isna(value) or value is None:
        return default
    try:
        return float(value) if isinstance(default, (int, float)) else value
    except (ValueError, TypeError):
        return default


def safe_str(value, default=''):
    if pd.isna(value) or value is None:
        return default
    return str(value)


def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'\*(.*?)\*', r'\1', text)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'&gt;.*?\n', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'[^\w\s.,!?;:-]', '', text)
    return text.strip()


def spacy_sent_tokenize(text):
    if not isinstance(text, str) or not text.strip():
        return []
    try:
        doc = nlp(text)
        return [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    except:
        return [s.strip() for s in text.split('.') if s.strip()]


def analyze_source_credibility(text):
    if not text or not isinstance(text, str):
        return 0.0

    bare_urls = re.findall(r'https?://[^\s<>\[\])\]]+', text)
    markdown_urls = re.findall(r'\[(?:[^\]]*)\]\((https?://[^)]+)\)', text)
    urls = list(set(bare_urls + markdown_urls))

    if not urls:
        return 0.0

    credibility_score = 0.0

    high_cred_domains = [
        '.edu', '.gov', 'scholar.google', 'jstor', 'pubmed',
        'nature.com', 'science.org', 'cell.com', 'nejm.org',
        'stanford.edu', 'harvard.edu', 'mit.edu',
        'who.int', 'nih.gov', 'cdc.gov'
    ]

    medium_cred_domains = [
        'reuters.com', 'ap.org', 'bbc.com', 'npr.org',
        'economist.com', 'wsj.com', 'nytimes.com',
        'washingtonpost.com', 'theguardian.com',
        'wikipedia.org', 'britannica.com'
    ]

    low_cred_indicators = [
        'blog', 'wordpress', 'medium.com', 'reddit.com',
        'youtube.com', 'twitter.com', 'facebook.com',
        'tiktok.com', 'instagram.com'
    ]

    for url in urls:
        url_lower = url.lower()
        if any(domain in url_lower for domain in high_cred_domains):
            credibility_score += 1.0
        elif any(domain in url_lower for domain in medium_cred_domains):
            credibility_score += 0.7
        elif any(indicator in url_lower for indicator in low_cred_indicators):
            credibility_score += 0.2
        else:
            credibility_score += 0.4

    return min(credibility_score / len(urls), 1.0)


def detect_persuasive_features(text):
    if not text or not isinstance(text, str):
        return 0.0

    text_lower = text.lower()
    score = 0.0

    # Penalize hostility
    aggressive_words = [
        'stupid', 'idiotic', 'ridiculous', 'absurd', 'moronic',
        'dumb', 'idiot', 'fool', 'pathetic', 'nonsense',
        'bullshit', 'crap', 'garbage'
    ]
    aggressive_count = sum(1 for w in aggressive_words if w in text_lower)
    score -= min(aggressive_count * 0.1, 0.3)

    # Reward direct engagement with OP
    direct_quotes = len(re.findall(r'>\s*\S+', text))
    score += min(direct_quotes * 0.15, 0.25)

    # Reward concrete examples
    specific_indicators = [
        r"for example", r"for instance", r"specifically",
        r"in particular", r"case in point", r"consider",
        r"take \w+", r"such as", r"like when"
    ]
    specific_count = sum(1 for p in specific_indicators if re.search(p, text_lower))
    score += min(specific_count * 0.1, 0.2)

    # Reward constructive framing
    constructive = [
        r"perhaps consider", r"might want to", r"have you thought",
        r"one way to think", r"another angle", r"reframe",
        r"another perspective", r"consider that", r"think about it"
    ]
    constructive_count = sum(1 for p in constructive if re.search(p, text_lower))
    score += min(constructive_count * 0.12, 0.25)

    # Length sweet spot
    word_count = len(text.split())
    if 100 <= word_count <= 400:
        score += 0.20
    elif 50 <= word_count < 100 or 400 < word_count <= 800:
        score += 0.10
    elif word_count > 1000:
        score -= 0.05

    # Reward personal experience
    personal_markers = [
        r"i've experienced", r"in my experience", r"i worked",
        r"i lived", r"as someone who", r"i was", r"i remember",
        r"when i", r"my own"
    ]
    personal_count = sum(1 for p in personal_markers if re.search(p, text_lower))
    score += min(personal_count * 0.08, 0.15)

    # Reward calm hedging
    hedging_patterns = [
        r"i think", r"perhaps", r"it seems", r"arguably",
        r"it could be", r"maybe", r"possibly", r"in my view"
    ]
    hedging_count = sum(1 for p in hedging_patterns if re.search(p, text_lower))
    score += min(hedging_count * 0.05, 0.15)

    # Reward concessions
    concession_patterns = [
        r"you're right that", r"valid point", r"i agree that",
        r"fair point", r"you make a good point", r"i can see",
        r"i understand why", r"that's true", r"good question"
    ]
    concession_count = sum(1 for p in concession_patterns if re.search(p, text_lower))
    score += min(concession_count * 0.12, 0.25)

    # Penalize excessive certainty
    certain_words = [
        r"obviously", r"clearly", r"definitely", r"absolutely",
        r"undoubtedly", r"without question", r"there is no doubt"
    ]
    certain_count = sum(1 for p in certain_words if re.search(p, text_lower))
    score -= min(certain_count * 0.05, 0.2)

    # Reward analogies
    analogy_patterns = [
        r"it's like", r"imagine if", r"similar to", r"comparable to",
        r"think of it as", r"analogous to", r"the same as"
    ]
    analogy_count = sum(1 for p in analogy_patterns if re.search(p, text_lower))
    score += min(analogy_count * 0.1, 0.2)

    return min(max(score, 0.0), 1.0)


def detect_argument_sophistication(text):
    if not text:
        return 0.0

    text_lower = text.lower()

    counter_patterns = [
        r"however,", r"that said", r"on the other hand", r"although",
        r"but consider", r"while.*?true", r"granted,",
        r"i understand.*?but", r"you have a point.*?but"
    ]
    counter_count = sum(1 for p in counter_patterns if re.search(p, text_lower))
    counter_score = min(counter_count * 0.2, 1.0)

    logic_patterns = [
        r"because", r"therefore", r"thus", r"consequently",
        r"as a result", r"this leads to", r"it follows that",
        r"the reason", r"that's why"
    ]
    logic_count = sum(1 for p in logic_patterns if re.search(p, text_lower))
    logic_score = min(logic_count * 0.15, 1.0)

    evidence_patterns = [
        r"according to", r"research shows", r"studies indicate",
        r"data suggests", r"evidence shows"
    ]
    evidence_count = sum(1 for p in evidence_patterns if re.search(p, text_lower))
    evidence_score = min(evidence_count * 0.2, 1.0)

    total = (
        0.50 * counter_score +
        0.30 * logic_score +
        0.20 * evidence_score
    )

    return min(total, 1.0)


def get_emotion_scores(text):
    if not text or pd.isna(text):
        return {
            'vader_compound': 0.0, 'vader_pos': 0.0,
            'vader_neg': 0.0, 'vader_neu': 0.0,
            'persuasive_emotion': 0.0
        }

    vader_scores = analyzer.polarity_scores(str(text))

    persuasive_emotions = [
        'understand', 'realize', 'believe', 'feel', 'think',
        'important', 'significant', 'crucial', 'essential'
    ]

    emotional_intensity = sum(1 for word in persuasive_emotions if word in str(text).lower())
    emotional_score = min(emotional_intensity / 15, 0.5)

    return {
        'vader_compound': vader_scores['compound'],
        'vader_pos': vader_scores['pos'],
        'vader_neg': vader_scores['neg'],
        'vader_neu': vader_scores['neu'],
        'persuasive_emotion': emotional_score
    }


def calculate_emotion_balance(text):
    emotion = get_emotion_scores(text)
    compound = emotion['vader_compound']

    if 0.0 <= compound <= 0.3:
        balance_score = 1.0
    elif -0.2 <= compound < 0.0:
        balance_score = 0.7
    elif 0.3 < compound <= 0.6:
        balance_score = 0.6
    elif -0.5 <= compound < -0.2:
        balance_score = 0.4
    elif compound > 0.6:
        balance_score = 0.3
    else:
        balance_score = 0.2

    return 0.7 * balance_score + 0.3 * emotion['persuasive_emotion']


def score_comment_persuasion(comment_text, evidence_score=None):
    if not comment_text or pd.isna(comment_text):
        return 0.0

    comment_text = str(comment_text)

    word_count = len(comment_text.split())
    if word_count < 20:
        return 0.0

    persuasive_features = detect_persuasive_features(comment_text)
    sophistication = detect_argument_sophistication(comment_text)

    if evidence_score is None:
        evidence_score = analyze_source_credibility(comment_text)

    emotion_balance = calculate_emotion_balance(comment_text)

    total = (
        0.40 * persuasive_features +
        0.25 * sophistication +
        0.20 * evidence_score +
        0.15 * emotion_balance
    )

    return round(min(max(total, 0.0), 1.0), 3)


def score_post_via_comments(post_id, comments_df):
    post_comments = comments_df[comments_df['post_id'] == post_id]

    empty_result = {
        'top_comment_score': 0.0,
        'top_5_avg_score': 0.0,
        'avg_comment_score': 0.0,
        'discussion_depth': 0.0,
        'substantial_comment_count': 0,
        'total_comments': len(post_comments)
    }

    if len(post_comments) == 0:
        return empty_result

    comment_scores = []
    for _, comment in post_comments.iterrows():
        text = safe_str(comment.get('comment_text', ''))
        if text:
            score = score_comment_persuasion(text)
            if score > 0:
                comment_scores.append(score)

    if not comment_scores:
        return empty_result

    sorted_scores = sorted(comment_scores, reverse=True)

    high_quality_count = sum(1 for s in comment_scores if s >= 0.20)
    discussion_depth = min(high_quality_count / 20, 1.0)

    if len(sorted_scores) >= 5:
        top_5_avg = np.mean(sorted_scores[:5])
    elif len(sorted_scores) >= 3:
        top_5_avg = np.mean(sorted_scores[:3]) * 0.85
    else:
        top_5_avg = sorted_scores[0] * 0.7 if sorted_scores else 0.0

    return {
        'top_comment_score': sorted_scores[0],
        'top_5_avg_score': top_5_avg,
        'avg_comment_score': np.mean(sorted_scores),
        'discussion_depth': discussion_depth,
        'substantial_comment_count': len(comment_scores),
        'total_comments': len(post_comments)
    }


def calculate_evidence_quality(post_id, comments_df, post_text=''):
    post_evidence = analyze_source_credibility(post_text) if post_text else 0.0

    post_comments = comments_df[comments_df['post_id'] == post_id]
    comment_evidence_scores = []

    for _, comment in post_comments.iterrows():
        text = safe_str(comment.get('comment_text', ''))
        if text:
            score = analyze_source_credibility(text)
            comment_evidence_scores.append(score)

    nonzero = [s for s in comment_evidence_scores if s > 0]
    if nonzero:
        avg_evidence = np.mean(nonzero)
        citation_rate = len(nonzero) / max(len(comment_evidence_scores), 1)
        comment_evidence = avg_evidence * citation_rate
    else:
        comment_evidence = 0.0

    combined = 0.3 * post_evidence + 0.7 * comment_evidence

    return {
        'post_evidence': post_evidence,
        'comment_evidence': comment_evidence,
        'combined_evidence': combined,
        'comments_with_links': len(nonzero)
    }


def calculate_engagement_quality(post_id, comments_df):
    post_comments = comments_df[comments_df['post_id'] == post_id]

    if len(post_comments) == 0:
        return 0.0

    valid_comments = post_comments[post_comments['comment_text'].apply(
        lambda x: not pd.isna(x) and str(x).strip() != ''
    )]

    if len(valid_comments) == 0:
        return 0.0

    avg_length = np.mean([len(safe_str(c['comment_text'])) for _, c in valid_comments.iterrows()])
    length_score = min(avg_length / 400, 0.3)

    if 'is_root_comment' in valid_comments.columns:
        non_root = valid_comments[valid_comments['is_root_comment'] == False]
        depth_score = min(len(non_root) / len(valid_comments) * 0.5, 0.3)
    else:
        depth_score = 0.0

    quality_keywords = {
        'thoughtful': ['interesting', 'good point', 'valid', 'thoughtful'],
        'constructive': ['however', 'but consider', 'what about'],
        'evidence': ['source', 'link', 'study', 'research'],
        'engagement': ['analysis', 'perspective', 'reasoning']
    }

    category_values = {
        'thoughtful': 0.1,
        'constructive': 0.15,
        'evidence': 0.15,
        'engagement': 0.05
    }

    quality_total = 0
    for _, comment in valid_comments.iterrows():
        body = safe_str(comment['comment_text']).lower()
        best_score = 0
        for category, keywords in quality_keywords.items():
            if any(keyword in body for keyword in keywords):
                best_score = max(best_score, category_values[category])
        quality_total += best_score

    quality_score = min(quality_total / max(len(valid_comments), 1) * 3, 0.4)

    return min(length_score + depth_score + quality_score, 1.0)


def analyze_posts(posts_df, comments_df):
    print("\nAnalyzing posts...")

    enhanced_posts = []

    for idx, post in posts_df.iterrows():
        post_id = safe_get(post, 'post_id', None)

        if post_id is None or pd.isna(post_id):
            print(f"   Warning: Invalid post_id at index {idx}, skipping")
            continue

        original_text = safe_str(post.get('original_post_text', ''))
        if not original_text:
            original_text = safe_str(post.get('post_text', ''))

        comment_metrics = score_post_via_comments(post_id, comments_df)
        evidence_metrics = calculate_evidence_quality(post_id, comments_df, original_text)
        engagement_score = calculate_engagement_quality(post_id, comments_df)

        clean_post = clean_text(original_text)
        sophistication_score = detect_argument_sophistication(clean_post)
        emotion_balance = calculate_emotion_balance(clean_post)

        predictive_score = (
            SCORING_WEIGHTS['top_comment_quality'] * comment_metrics['top_5_avg_score'] +
            SCORING_WEIGHTS['discussion_depth'] * comment_metrics['discussion_depth'] +
            SCORING_WEIGHTS['evidence_quality'] * evidence_metrics['combined_evidence'] +
            SCORING_WEIGHTS['engagement_quality'] * engagement_score +
            SCORING_WEIGHTS['sophistication'] * sophistication_score +
            SCORING_WEIGHTS['emotion_balance'] * emotion_balance
        )
        predictive_score = round(min(predictive_score, 1.0), 3)

        delta_count = safe_get(post, 'delta_count', 0)
        delta_score = min(delta_count * 0.25, 1.0)
        composite_score = round(0.7 * predictive_score + 0.3 * delta_score, 3)

        emotion_data = get_emotion_scores(clean_post)

        enhanced_post = post.to_dict()
        enhanced_post.update({
            'predictive_persuasion_score': predictive_score,
            'composite_persuasion_score': composite_score,
            'persuasion_score': predictive_score,
            'top_comment_score': comment_metrics['top_comment_score'],
            'top_5_avg_score': comment_metrics['top_5_avg_score'],
            'avg_comment_score': comment_metrics['avg_comment_score'],
            'discussion_depth': comment_metrics['discussion_depth'],
            'substantial_comment_count': comment_metrics['substantial_comment_count'],
            'evidence_quality_score': evidence_metrics['combined_evidence'],
            'post_evidence_score': evidence_metrics['post_evidence'],
            'comment_evidence_score': evidence_metrics['comment_evidence'],
            'comments_with_links': evidence_metrics['comments_with_links'],
            'comment_engagement_score': engagement_score,
            'argument_sophistication_score': sophistication_score,
            'emotion_balance_score': emotion_balance,
            'vader_compound': emotion_data['vader_compound'],
            'vader_positive': emotion_data['vader_pos'],
            'vader_negative': emotion_data['vader_neg'],
            'vader_neutral': emotion_data['vader_neu'],
            'has_deltas': delta_count > 0,
            'has_substantial_discussion': comment_metrics['substantial_comment_count'] >= 20,
        })

        enhanced_posts.append(enhanced_post)
        print(f"   Post {idx + 1}/{len(posts_df)}: score={predictive_score:.3f}, deltas={int(delta_count)}, top_comment={comment_metrics['top_comment_score']:.3f}")

    return pd.DataFrame(enhanced_posts)


def analyze_comments(comments_df):
    print(f"\nAnalyzing {len(comments_df)} comments...")

    enhanced_comments = []
    skipped_count = 0

    for idx, comment in comments_df.iterrows():
        comment_text = safe_str(comment.get('comment_text', ''))

        if not comment_text or len(comment_text.split()) < 20:
            skipped_count += 1
            continue

        evidence = analyze_source_credibility(comment_text)
        persuasion_score = score_comment_persuasion(comment_text, evidence_score=evidence)
        persuasive_features = detect_persuasive_features(comment_text)
        sophistication = detect_argument_sophistication(comment_text)
        emotion_balance = calculate_emotion_balance(comment_text)
        emotion = get_emotion_scores(comment_text)

        enhanced_comment = comment.to_dict()
        enhanced_comment.update({
            'persuasion_score': persuasion_score,
            'persuasive_features_score': persuasive_features,
            'sophistication_score': sophistication,
            'evidence_quality_score': evidence,
            'emotion_balance_score': emotion_balance,
            'vader_compound': emotion['vader_compound'],
            'vader_positive': emotion['vader_pos'],
            'vader_negative': emotion['vader_neg'],
            'vader_neutral': emotion['vader_neu'],
            'is_substantial': len(comment_text.split()) >= 20,
            'is_high_quality': persuasion_score >= 0.2,
        })

        enhanced_comments.append(enhanced_comment)

        if (idx + 1) % 1000 == 0:
            print(f"   Processed {idx + 1}/{len(comments_df)} comments (skipped {skipped_count})")

    print(f"   Completed: {len(enhanced_comments)} comments analyzed, {skipped_count} skipped")
    return pd.DataFrame(enhanced_comments)


def load_data(posts_file='cmv_posts_analysis.csv', comments_file='cmv_comments_analysis.csv'):
    print(f"Loading data...")

    try:
        posts_df = pd.read_csv(posts_file)
        print(f"   Loaded {len(posts_df)} posts")
    except FileNotFoundError:
        print(f"Error: Could not find {posts_file}")
        return None, None

    try:
        comments_df = pd.read_csv(comments_file)
        print(f"   Loaded {len(comments_df)} comments")
    except FileNotFoundError:
        print(f"Error: Could not find {comments_file}")
        return None, None

    return posts_df, comments_df


def save_results(posts_df, comments_df):
    posts_df.to_csv('cmv_posts_results.csv', index=False)
    print(f"Posts saved to 'cmv_posts_results.csv'")

    comments_df.to_csv('cmv_comments_results.csv', index=False)
    print(f"Comments saved to 'cmv_comments_results.csv'")


def display_results(posts_df, comments_df):
    print("\nPersuasion Analysis Results")
    print("-" * 50)

    print(f"\nDataset:")
    print(f"   Posts: {len(posts_df):,}")
    print(f"   Comments: {len(comments_df):,}")
    print(f"   Substantial comments: {comments_df['is_substantial'].sum():,}")
    print(f"   High-quality comments: {comments_df['is_high_quality'].sum():,}")

    print(f"\nPredictive Scores:")
    print(f"   Average: {posts_df['predictive_persuasion_score'].mean():.3f}")
    print(f"   Highest: {posts_df['predictive_persuasion_score'].max():.3f}")
    print(f"   Lowest: {posts_df['predictive_persuasion_score'].min():.3f}")
    print(f"   Std dev: {posts_df['predictive_persuasion_score'].std():.3f}")

    print(f"\nDeltas:")
    print(f"   Total: {posts_df['delta_count'].sum()}")
    print(f"   Posts with deltas: {(posts_df['delta_count'] > 0).sum()}/{len(posts_df)}")
    print(f"   Avg per post: {posts_df['delta_count'].mean():.1f}")

    print(f"\nComment Quality:")
    print(f"   Avg top comment score: {posts_df['top_comment_score'].mean():.3f}")
    print(f"   Avg top-5 score: {posts_df['top_5_avg_score'].mean():.3f}")
    print(f"   Avg substantial comments per post: {posts_df['substantial_comment_count'].mean():.0f}")

    print(f"\nTop Posts:")
    top_posts = posts_df.nlargest(5, 'predictive_persuasion_score')[
        ['title', 'predictive_persuasion_score', 'delta_count', 'top_comment_score']
    ]
    for i, (_, row) in enumerate(top_posts.iterrows(), 1):
        title = str(row['title'])[:55] + "..." if len(str(row['title'])) > 55 else str(row['title'])
        print(f"   {i}. [{row['predictive_persuasion_score']:.3f}] {title}")
        print(f"      Deltas: {row['delta_count']:.0f} | Top comment: {row['top_comment_score']:.3f}")

    print(f"\nCorrelations with Delta Count:")
    corr_predictive = posts_df['predictive_persuasion_score'].corr(posts_df['delta_count'])
    corr_top_comment = posts_df['top_comment_score'].corr(posts_df['delta_count'])
    corr_top5 = posts_df['top_5_avg_score'].corr(posts_df['delta_count'])
    corr_evidence = posts_df['evidence_quality_score'].corr(posts_df['delta_count'])
    corr_engagement = posts_df['comment_engagement_score'].corr(posts_df['delta_count'])
    corr_discussion = posts_df['discussion_depth'].corr(posts_df['delta_count'])

    print(f"   Predictive score: {corr_predictive:+.3f}")
    print(f"   Top comment score: {corr_top_comment:+.3f}")
    print(f"   Top-5 avg score: {corr_top5:+.3f}")
    print(f"   Evidence quality: {corr_evidence:+.3f}")
    print(f"   Engagement quality: {corr_engagement:+.3f}")
    print(f"   Discussion depth: {corr_discussion:+.3f}")


if __name__ == "__main__":
    print("CMV Persuasion Analysis")
    print("-" * 50)

    posts_df, comments_df = load_data(
        posts_file='cmv_posts_analysis.csv',
        comments_file='cmv_comments_analysis.csv'
    )

    if posts_df is None or comments_df is None:
        print("\nFailed to load data files.")
        exit(1)

    analyzed_comments = analyze_comments(comments_df)
    analyzed_posts = analyze_posts(posts_df, analyzed_comments)

    save_results(analyzed_posts, analyzed_comments)
    display_results(analyzed_posts, analyzed_comments)

    print("\nAnalysis complete.")

CMV Persuasion Analysis
--------------------------------------------------
Loading data...
   Loaded 10 posts
   Loaded 14034 comments

Analyzing 14034 comments...
   Processed 1000/14034 comments (skipped 297)
   Processed 3000/14034 comments (skipped 872)
   Processed 4000/14034 comments (skipped 1192)
   Processed 5000/14034 comments (skipped 1506)
   Processed 6000/14034 comments (skipped 1725)
   Processed 7000/14034 comments (skipped 1955)
   Processed 8000/14034 comments (skipped 2233)
   Processed 9000/14034 comments (skipped 2598)
   Processed 10000/14034 comments (skipped 2914)
   Processed 11000/14034 comments (skipped 3168)
   Processed 13000/14034 comments (skipped 3574)
   Processed 14000/14034 comments (skipped 3838)
   Completed: 10184 comments analyzed, 3850 skipped

Analyzing posts...
   Post 1/10: score=0.453, deltas=4, top_comment=0.274
   Post 2/10: score=0.373, deltas=4, top_comment=0.259
   Post 3/10: score=0.324, deltas=11, top_comment=0.291
   Post 4/10: score=